In [0]:
# Databricks notebook source


# 03 - Gold: Comportamento de Gorjeta

Camada analítica final, respondendo: **o que influencia o percentual de gorjeta em corridas de
táxi, e como isso varia por zona, horário, tipo de dia e distância da corrida?**

## Limitação de dados documentada

No dataset da NYC TLC, `tip_amount` só é confiável para pagamentos em **cartão de crédito**
(`payment_type = 1`). Gorjetas em dinheiro não passam pelo sistema de pagamento eletrônico do táxi e
ficam registradas como `0`, mesmo quando o passageiro efetivamente deu gorjeta. Ou seja: um `0` nessa
coluna pode significar "não deu gorjeta" ou "deu gorjeta em dinheiro, e o dado não existe".
Decisão tomada aqui: **não filtrar** os dados só para cartão — manter todos os pagamentos, mas expor
uma métrica (`pct_card_payment`) que mostra, para cada grupo (zona/hora/etc.), qual fração das
corridas foi paga em cartão. Isso deixa explícito, linha a linha, quando um `avg_tip_pct` baixo é
"gorjeta realmente baixa" versus "muita corrida em dinheiro nesse grupo, dado não confiável".

## Estratégia de carga

Diferente do Bronze/Silver (incrementais por mês), o Gold é um **recompute completo** a cada
execução — processa tudo que existir na Silver até o momento. É uma escolha comum em camadas
analíticas: o volume agregado é pequeno, e os rankings/comparações fazem mais sentido olhando o
conjunto inteiro de dados disponível, não só o mês mais recente.
 

## Configuração

In [0]:
CATALOG = "nyc_taxi"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
 
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.trips_silver"
GOLD_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.tip_behavior_gold"

## Query analítica

Estrutura em 2 CTEs + seleção final:

1. **`base`** — prepara colunas derivadas por linha (tipo de dia, faixa de distância, % de gorjeta
   individual da corrida)
2. **`aggregated`** — agrupa por zona/hora/tipo de dia/faixa de distância, calculando as métricas
3. **Seleção final** — aplica window functions sobre o resultado agregado, comparando cada grupo
   contra a média da sua própria hora
 

In [0]:
gold_df = spark.sql(f"""
WITH base AS (
    SELECT
        pu_location_id,
        pickup_hour,
        CASE WHEN pickup_dow IN (1, 7) THEN 'fim_de_semana' ELSE 'dia_util' END AS day_type,
        CASE
            WHEN trip_distance <= 2 THEN 'curta'
            WHEN trip_distance <= 6 THEN 'media'
            ELSE 'longa'
        END AS distance_bucket,
        payment_type,
        fare_amount,
        CASE WHEN fare_amount > 0 THEN tip_amount / fare_amount ELSE NULL END AS tip_pct
    FROM {SILVER_TABLE}
),
 
aggregated AS (
    SELECT
        pu_location_id,
        pickup_hour,
        day_type,
        distance_bucket,
        COUNT(*) AS trip_count,
        ROUND(AVG(CAST(CASE WHEN payment_type = 1 THEN 1 ELSE 0 END AS DOUBLE)) * 100, 2) AS pct_card_payment,
        ROUND(AVG(tip_pct) * 100, 2) AS avg_tip_pct,
        ROUND(percentile_approx(tip_pct, 0.5) * 100, 2) AS median_tip_pct
    FROM base
    GROUP BY pu_location_id, pickup_hour, day_type, distance_bucket
    HAVING COUNT(*) >= 20
)
 
SELECT
    pu_location_id,
    pickup_hour,
    day_type,
    distance_bucket,
    trip_count,
    pct_card_payment,
    avg_tip_pct,
    median_tip_pct,
    RANK() OVER (PARTITION BY pickup_hour, day_type ORDER BY avg_tip_pct DESC) AS tip_rank_in_hour,
    ROUND(AVG(avg_tip_pct) OVER (PARTITION BY pickup_hour, day_type), 2) AS hour_avg_tip_pct,
    ROUND(avg_tip_pct - AVG(avg_tip_pct) OVER (PARTITION BY pickup_hour, day_type), 2) AS tip_pct_diff_from_hour_avg,
    current_timestamp() AS _gold_processed_at
FROM aggregated
ORDER BY pickup_hour, day_type, tip_rank_in_hour
""")
 
row_count = gold_df.count()
print(f"Linhas geradas para o Gold: {row_count}")
 

## Gravação (recompute completo)
 

In [0]:
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)
 
print(f"Gold atualizado: {row_count} linhas.")
 

## Verificação rápida — top 10 zonas com maior gorjeta média em corridas de fim de semana à noite
 

In [0]:
from pyspark.sql import functions as F
 
display(
    spark.table(GOLD_TABLE)
    .filter((F.col("day_type") == "fim_de_semana") & (F.col("pickup_hour").between(20, 23)))
    .orderBy("tip_rank_in_hour")
    .limit(10)
)